A script implementing BICePs to reweight populations for a simple three state
toy model system.  Here, our prior comes from random generation of the Boltzmann
distribution and reweighting is performed using two experimental observables both set to 0.0 A.U.

For more details about this toy model systema and visual aids, please refer to
this notebook: `examples/enforcing_uniform_reference.ipynb`

In [1]:
import sys, os
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
import pandas as pd
from sklearn import metrics
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import biceps
from biceps.PosteriorSampler import u_kln_and_states_kn
from pymbar import MBAR

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



In [2]:
class Data:
    def __init__(self, array_list):
        self.array_list = array_list

    def save(self, filename):
        with open(filename, 'wb') as f:
            pickle.dump(self.array_list, f)

    @classmethod
    def load(cls, filename):
        with open(filename, 'rb') as f:
            array_list = pickle.load(f)
        return cls(array_list)

In [3]:
# Write noe files:{{{
def write_noe_files(weights, x, exp, dir):
    for i in range(len(weights)):
        model = pd.read_pickle("template.noe")
        _model = pd.DataFrame()
        for j in range(len(exp)):
            model["restraint_index"], model["model"], model["exp"] = 1+j, x[i][j], 0.0
            #model["restraint_index"], model["model"], model["exp"] = 1+j, x[i][j], exp[j]
            #_model = _model.append(model, ignore_index=True)
            _model = pd.concat([_model,model], ignore_index=True)
        _model.to_pickle(dir+"/%s.noe"%i)

#:}}}

###### Parameters #######

In [4]:
nStates,Nd = 76,1 # 76 distance with 1 NOE observables
n_xis,n_lambdas,nreplicas,nsteps,change_Nr_every,swap_every=1,2,1,1000000,0,0
multiprocess=4
σ_prior=0.161 # 0.08, 0.16
stat_model,data_uncertainty="Students","single"
data_likelihood = "gaussian" #"log normal" # "gaussian"

write_every = 10
attempt_move_state_every = 1
attempt_move_sigma_every = 1

Make output directories

In [5]:
state_dir = f"{nStates}_state"
biceps.toolbox.mkdir(state_dir)

datapoints_dir = f"{state_dir}/{nStates}_state_{Nd}_datapoints"
biceps.toolbox.mkdir(datapoints_dir)

dir = f"{datapoints_dir}/Prior_error_{σ_prior}"
biceps.toolbox.mkdir(dir)

## Loaded the population

In [6]:
clustering = pd.read_csv(f"../clustering/cluster_percentages.csv")

populations = np.array(clustering["Population"])
populations.shape

(92,)

In [7]:
energies = -np.log(populations)
energies.shape

(92,)

## Load the Prior Model (From MD) Calculated NOE distances 

In [8]:
md_distances = pd.read_csv(f"../clustering/md_distances.csv")

forward_model_data = np.array(md_distances)
forward_model_data.shape

(92, 76)

## Load the Refer NMR (Experimental Measurement)

In [9]:
restraints_table = {
    'weak': 5,
    'medium': 3.5,
    'strong': 2.5
}

dist_res_file = '../../../../utils/nspe_7_1_restraints.csv'
df_dist_res = pd.read_csv(dist_res_file)

for i, row in df_dist_res.iterrows():
    df_dist_res.at[i, df_dist_res.columns[2]] = restraints_table[row[2]]  # Correct mapping

# Make a look up table for intensity 
df_dist_res
dist_res = df_dist_res.values.tolist()
dist_res[:5]

[[160, 165, 5], [160, 166, 5], [161, 165, 5], [161, 166, 5], [137, 165, 5]]

In [10]:
np.shape(dist_res)

(76, 3)

In [11]:
experiment = np.array([row[-1] for row in dist_res])
experiment[:5]


array([5., 5., 5., 5., 5.])

## Write the NOE files 

In [12]:
data_dir = f"{dir}/NOE"
biceps.toolbox.mkdir(data_dir)

write_noe_files(weights=energies, x=forward_model_data, exp=experiment, dir=data_dir)

## Load the input data

In [13]:
input_data = biceps.toolbox.sort_data(data_dir)
print(f"Input data: {biceps.toolbox.list_extensions(input_data)}")
forward_model_data = np.array([pd.read_pickle(i)["model"].to_numpy() for i in biceps.toolbox.get_files(f"{data_dir}/*.noe")])
experiment = np.array([pd.read_pickle(i)["exp"].to_numpy() for i in biceps.toolbox.get_files(f"{data_dir}/0.noe")])[0]


Input data: ['.noe']


In [14]:
outdir = f'{dir}/{stat_model}_{data_uncertainty}_sigma/{nsteps}_steps_{nreplicas}_replicas_{n_lambdas}_lam__swap_every_{swap_every}'
biceps.toolbox.mkdir(outdir)
print(f"nSteps of sampling: {nsteps}\nnReplicas: {nreplicas}")
lambda_values = np.linspace(0.0, 1.0, n_lambdas)

nSteps of sampling: 1000000
nReplicas: 1


In [15]:
sigMin,sigMax,dsig = 0.001,200,1.02
arr = np.exp(np.arange(np.log(sigMin), np.log(sigMax), np.log(dsig)))
l = len(arr)
sigma_index = round(l*0.73)

In [16]:
beta,beta_index=(1., 2.0, 1),0
_arr = np.linspace(*beta)
_l = len(_arr)
print("Alpha starts here: ",_arr[beta_index])
phi,phi_index=(1., 2.0, 1),0
gamma,gamma_index=(1.0, 2.0, np.e),0

Alpha starts here:  1.0


In [17]:
options = [dict(ref="uniform", stat_model=stat_model,
            sigma=(sigMin, sigMax, dsig), sigma_index=sigma_index, gamma=gamma,
            beta=beta, beta_index=beta_index, phi=phi, phi_index=phi_index,
            data_uncertainty=data_uncertainty, data_likelihood=data_likelihood,
            )]
print(pd.DataFrame(options))


       ref stat_model               sigma  sigma_index  \
0  uniform   Students  (0.001, 200, 1.02)          450   

                           gamma           beta  beta_index            phi  \
0  (1.0, 2.0, 2.718281828459045)  (1.0, 2.0, 1)           0  (1.0, 2.0, 1)   

   phi_index data_uncertainty data_likelihood  
0          0           single        gaussian  


In [18]:
ensemble = biceps.ExpandedEnsemble(lambda_values=lambda_values, energies=energies)
ensemble.initialize_restraints(input_data, options, verbose=1)
print("ensemble.expanded_values = ",ensemble.expanded_values)

Time to initalize restraints: 0.21s
ensemble.expanded_values =  [(0.0, 1.0), (1.0, 1.0)]


In [19]:
sampler = biceps.PosteriorSampler(ensemble, nreplicas, change_Nr_every, write_every=write_every)
sampler.sample(nsteps, attempt_lambda_swap_every=swap_every, swap_sigmas=1,
        attempt_move_state_every=attempt_move_state_every,
        attempt_move_sigma_every=attempt_move_sigma_every,
        verbose=0, progress=1, multiprocess=True, capture_stdout=0)

 ██████████████████████████████▏ 100.0% [1000000/1000000 | 83.5 kHz | 1 | 0s | 12s] MCMC 


In [20]:
expanded_values = sampler.expanded_values
A = biceps.Analysis(sampler, outdir=outdir, nstates=len(energies), MBAR=True, multiprocess=False, capture_stdout=0)
A.plot(plottype="step", figsize=(12,14), figname=f"BICePs.pdf", pad=0.35, plot_all_distributions=1)
plt.show()
A.plot_energy_trace()
plt.show()
BS, pops = A.f_df, A.P_dP[:,len(expanded_values[:])-1]
BS /= sampler.nreplicas
K = len(expanded_values[:])-1
pops_std = A.P_dP[:,2*K]
print(f"Predicted populatins: {pops}")

These states have not been sampled:
 [13 26 36 66]
These states have not been sampled:
 [ 2 11 17 20 24 25 26 36 38 40 42 44 46 47 49 50 56 59 60 61 63 64 66 69
 71 73 76 78 82 83 87 89]
                               ▏  0.0% [   0/100000 | 0.0 Hz | 1 | infs | 0s] u_kln 


******* JAX 64-bit mode is now on! *******
*     JAX is now set to 64-bit mode!     *
*   This MAY cause problems with other   *
*      uses of JAX in the same code.     *
******************************************



 ██████████████████████████████▏ 100.0% [100000/100000 | 367.7 kHz | 1 | 0s | 0s] u_kln 
Time for MBAR: 3.534 s
Writing 76_state/76_state_1_datapoints/Prior_error_0.161/Students_single_sigma/1000000_steps_1_replicas_2_lam__swap_every_0/BS.dat...
Writing 76_state/76_state_1_datapoints/Prior_error_0.161/Students_single_sigma/1000000_steps_1_replicas_2_lam__swap_every_0/populations.dat...
Top 9 states: [23, 6, 22, 10, 15, 74, 0, 9, 7]
Top 9 populations: [0.00108734 0.00122595 0.00362681 0.00542264 0.0149709  0.04798371
 0.18865778 0.24667904 0.47927844]
nplots =  1


/var/folders/d8/y2dvs1ln1gjcwccrkvtffr240000gn/T/ipykernel_91050/3964779525.py:4: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Predicted populatins: [1.88657782e-01 1.04798103e-03 1.04247793e-05 7.79212308e-05
 1.50718434e-04 6.45729009e-05 1.22595254e-03 4.79278443e-01
 1.04690974e-03 2.46679035e-01 5.42263771e-03 1.14853348e-07
 1.29002119e-05 6.44977921e-05 6.37826283e-04 1.49708968e-02
 8.01171023e-05 8.44214342e-06 8.34456205e-04 7.37384881e-04
 4.53555606e-06 3.81551865e-05 3.62680962e-03 1.08733993e-03
 5.06353945e-06 1.08317581e-05 0.00000000e+00 3.48301281e-04
 2.98451610e-05 2.93472957e-04 8.27145085e-06 2.19224182e-05
 9.24066572e-06 1.43121147e-04 1.08257494e-04 2.01821462e-05
 0.00000000e+00 5.34569503e-04 1.57622865e-06 3.71600621e-04
 4.13183911e-06 7.35885241e-04 4.32803932e-07 1.54063113e-04
 3.58659256e-08 2.30580850e-04 5.15360296e-07 3.83762754e-08
 9.22486830e-05 5.76310685e-06 1.47680843e-07 1.06787628e-05
 7.97287420e-05 1.67368525e-05 9.34635200e-05 7.74185689e-05
 1.17393787e-06 2.29095733e-06 5.08895892e-05 9.33579018e-06
 1.87818506e-06 3.34579698e-07 5.05521487e-06 9.59925482e-07
 1

In [21]:
pops.shape

# Convert to DataFrame
df_predicted_population = pd.DataFrame(pops)

# Save to CSV
df_predicted_population.to_csv("../clustering/predicted_population_biceps.csv", index=False)

In [22]:
most_populated_index = np.argmax(pops)
most_populated_value = pops[most_populated_index]

print(f"Most populated state: {most_populated_index} with value {most_populated_value}")


Most populated state: 7 with value 0.479278442799456


In [23]:
top5_indices = np.argsort(pops)[-5:][::-1]  # Sort, take last 5, reverse for descending order
top5_values = pops[top5_indices]

for i, (idx, val) in enumerate(zip(top5_indices, top5_values), 1):
    print(f"Top {i}: index = {idx}, population = {val:.4f}")

top5_indices

Top 1: index = 7, population = 0.4793
Top 2: index = 9, population = 0.2467
Top 3: index = 0, population = 0.1887
Top 4: index = 74, population = 0.0480
Top 5: index = 15, population = 0.0150


array([ 7,  9,  0, 74, 15])